In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2018primate")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "2018pone_exp1.csv")
complete_path_2 = os.path.join(original_data_pathway, "2018pone_exp2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1 = df1.assign(experiment='1')

df2 = pd.read_csv(complete_path_2)
df2 = df2.assign(experiment='2')
df2.rename(columns={"Action Target":"in-hand_action_targets",
              "Distal Target": "distal_action_targets"}, inplace=True)
# df2.columns

In [3]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        "species":"species_original",
        "group":"group_id"}, inplace=True)
    x['ape'] = x['ape'].str.rstrip()
    x['study_id']="kano2018primate"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)



In [4]:
# fulldf['species_original'].replace('chimp', 'chimpanzee', inplace=True, regex=True)

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')



In [6]:
comp_path_name_errors = os.path.join(pathway_gen, "kano2018primate_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "kano2018primate_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)  
apedf = apedf.values.tolist()

for x,y,k in apedf:
    fulldf.loc[fulldf.ape == x, ['species', 'sex']] = y,k 

In [7]:
fulldf = fulldf[~fulldf['species_original'].isin(['human'])]
fulldf = fulldf[~fulldf['group_id'].isin(['humexpert', 'humlay', 'hupresch'])]

In [8]:
# spe_2=[]  
# for index, row in fulldf.iterrows():
#     if not pd.isna(row['species']):
#         spe_2.append(row['species'])
#     else:
#         spe_2.append(row['species_original'])
# fulldf = fulldf.assign(species=spe_2)

In [9]:
spe_2=[]  
# for index, row in fulldf.iterrows():
#     if not pd.isna(row['species']):
#         spe_2.append(row['species'])
#     elif "chimp" in str(row['spe2']):
#             spe_2.append('chimpanzee')
#     else: 
#         spe_2.append('')
# fulldf = fulldf.assign(species=spe_2)

In [10]:
import re
replace_1=re.compile('(\ |\*)')
fulldf.columns = fulldf.columns.str.replace(replace_1, '_')
fulldf['group_id'].replace('chimp', '', inplace=True, regex=True)
# fulldf.columns
fulldf.rename(columns={"ape":"participant"}, inplace=True)

In [11]:

fulldf=fulldf[['study_id','experiment',   'participant', 'sex', 'species','group_id', 'eye', 'mouth', 'head',
       'action_target',   'in-hand_action_targets', 'distal_action_targets']]



In [12]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'kano2018primate_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'kano2018primate_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)